# LR1. SF Bay Area Bike Share analysis with Apache Spark

Data files: `trips.csv`, `stations.csv`.

Tasks:
1. Find the bike with the maximum total ride duration.
2. Find the largest geodesic distance between stations.
3. Find the route of the bike with the maximum total ride duration through stations.
4. Find the number of bikes in the system.
5. Find users who spent more than 3 hours on trips.

## 1. Install PySpark in Google Colab

This cell installs PySpark only if it is missing.

In [1]:
try:
    import pyspark
    print(f"PySpark already installed: {pyspark.__version__}")
except ModuleNotFoundError:
    %pip -q install pyspark
    import pyspark
    print(f"PySpark installed: {pyspark.__version__}")

PySpark already installed: 4.0.2


## 2. Create Spark session

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("LR1 Bike Share")
    .master("local[*]")
    .getOrCreate()
)

spark.version

'4.0.2'

## 3. Load data

The notebook supports two modes:
- local repository mode: files are in `data/trips.csv` and `data/stations.csv`;
- Colab mode: if files are not found, upload `trips.csv` and `stations.csv` when prompted.

In [3]:
from pathlib import Path


def find_file(candidates):
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    return None

trips_path = find_file([
    "../data/trips.csv",
    "data/trips.csv",
    "/content/trips.csv",
    "trips.csv",
])
stations_path = find_file([
    "../data/stations.csv",
    "data/stations.csv",
    "/content/stations.csv",
    "stations.csv",
])

if trips_path is None or stations_path is None:
    try:
        from google.colab import files
        print("Upload trips.csv and stations.csv")
        files.upload()
    except ModuleNotFoundError as exc:
        raise FileNotFoundError(
            "trips.csv and stations.csv were not found. Put them into data/ or near the notebook."
        ) from exc

    trips_path = find_file(["/content/trips.csv", "trips.csv"])
    stations_path = find_file(["/content/stations.csv", "stations.csv"])

if trips_path is None or stations_path is None:
    raise FileNotFoundError("trips.csv and stations.csv were not found after upload")

print(f"trips: {trips_path}")
print(f"stations: {stations_path}")

trips_raw = spark.read.csv(str(trips_path), header=True, inferSchema=True)
stations_raw = spark.read.csv(str(stations_path), header=True, inferSchema=True)

Upload trips.csv and stations.csv


Saving stations.csv to stations.csv
Saving trips.csv to trips.csv
trips: /content/trips.csv
stations: /content/stations.csv


## 4. Initial checks

Check schema, sample rows, row counts, and null values in key fields.

In [4]:
trips_raw.printSchema()
stations_raw.printSchema()

print("trips count:", trips_raw.count())
print("stations count:", stations_raw.count())

trips_raw.show(5, truncate=False)
stations_raw.show(5, truncate=False)

root
 |-- id: integer (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_date: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- end_date: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dock_count: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- installation_date: string (nullable = true)

trips count: 669959
stations count: 70
+----+--------+---------------+------------------------+----------------+---------------+------------------------+--------------+-------+-----------------+--------+
|id  |duration|start_date     |st

In [5]:
key_trip_columns = [
    "duration",
    "start_date",
    "start_station_id",
    "end_date",
    "end_station_id",
    "bike_id",
    "zip_code",
]

trips_raw.select([
    F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), c)).alias(c)
    for c in key_trip_columns
]).show(truncate=False)

+--------+----------+----------------+--------+--------------+-------+--------+
|duration|start_date|start_station_id|end_date|end_station_id|bike_id|zip_code|
+--------+----------+----------------+--------+--------------+-------+--------+
|1       |1         |0               |0       |0             |0      |6619    |
+--------+----------+----------------+--------+--------------+-------+--------+



## 5. Prepare data

`trips.csv` contains missing `duration` and date values. Cast fields to correct types and filter rows without key fields.

In [6]:
trips = (
    trips_raw
    .withColumn("trip_id", F.col("id").cast("long"))
    .withColumn("duration", F.col("duration").cast("long"))
    .withColumn("start_station_id", F.col("start_station_id").cast("int"))
    .withColumn("end_station_id", F.col("end_station_id").cast("int"))
    .withColumn("bike_id", F.col("bike_id").cast("int"))
    .withColumn("start_ts", F.to_timestamp("start_date", "M/d/yyyy H:mm"))
    .withColumn("end_ts", F.to_timestamp("end_date", "M/d/yyyy H:mm"))
)

valid_trips = trips.filter(
    F.col("duration").isNotNull()
    & F.col("bike_id").isNotNull()
    & F.col("start_station_id").isNotNull()
    & F.col("end_station_id").isNotNull()
)

valid_trips_with_dates = valid_trips.filter(
    F.col("start_ts").isNotNull() & F.col("end_ts").isNotNull()
)

stations = (
    stations_raw
    .withColumn("station_id", F.col("id").cast("int"))
    .withColumn("lat", F.col("lat").cast("double"))
    .withColumn("long", F.col("long").cast("double"))
    .filter(F.col("station_id").isNotNull() & F.col("lat").isNotNull() & F.col("long").isNotNull())
)

print("valid trips:", valid_trips.count())
print("valid trips with dates:", valid_trips_with_dates.count())
print("valid stations:", stations.count())

valid trips: 669958
valid trips with dates: 669957
valid stations: 70


## Task 1. Bike with maximum total ride duration

Group trips by `bike_id` and calculate total ride duration.

In [7]:
bike_total_duration = (
    valid_trips
    .groupBy("bike_id")
    .agg(
        F.sum("duration").alias("total_duration_sec"),
        F.count("*").alias("trips_count"),
    )
    .withColumn("total_duration_hours", F.round(F.col("total_duration_sec") / 3600, 2))
    .orderBy(F.desc("total_duration_sec"))
)

max_duration_bike = bike_total_duration.limit(1)
max_duration_bike.show(truncate=False)

+-------+------------------+-----------+--------------------+
|bike_id|total_duration_sec|trips_count|total_duration_hours|
+-------+------------------+-----------+--------------------+
|535    |18611693          |1328       |5169.91             |
+-------+------------------+-----------+--------------------+



## Task 2. Largest geodesic distance between stations

Use the haversine formula. The stations table is small, so a self cross join is acceptable.

In [8]:
earth_radius_km = 6371.0

s1 = stations.select(
    F.col("station_id").alias("station_id_1"),
    F.col("name").alias("station_name_1"),
    F.col("lat").alias("lat_1"),
    F.col("long").alias("long_1"),
)
s2 = stations.select(
    F.col("station_id").alias("station_id_2"),
    F.col("name").alias("station_name_2"),
    F.col("lat").alias("lat_2"),
    F.col("long").alias("long_2"),
)

station_pairs = s1.crossJoin(s2).filter(F.col("station_id_1") < F.col("station_id_2"))

lat1 = F.radians(F.col("lat_1"))
lat2 = F.radians(F.col("lat_2"))
dlat = F.radians(F.col("lat_2") - F.col("lat_1"))
dlon = F.radians(F.col("long_2") - F.col("long_1"))

a = F.pow(F.sin(dlat / 2), 2) + F.cos(lat1) * F.cos(lat2) * F.pow(F.sin(dlon / 2), 2)
c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))

max_station_distance = (
    station_pairs
    .withColumn("distance_km", F.round(F.lit(earth_radius_km) * c, 3))
    .orderBy(F.desc("distance_km"))
    .limit(1)
)

max_station_distance.select(
    "station_id_1", "station_name_1",
    "station_id_2", "station_name_2",
    "distance_km",
).show(truncate=False)

+------------+--------------------------+------------+----------------------+-----------+
|station_id_1|station_name_1            |station_id_2|station_name_2        |distance_km|
+------------+--------------------------+------------+----------------------+-----------+
|16          |SJSU - San Salvador at 9th|60          |Embarcadero at Sansome|69.921     |
+------------+--------------------------+------------+----------------------+-----------+



## Task 3. Route of the bike with maximum total ride duration

Take `bike_id` from Task 1 and show all trips of this bike in chronological order.

In [9]:
max_bike_id = max_duration_bike.collect()[0]["bike_id"]
print("bike_id with maximum total duration:", max_bike_id)

max_bike_route = (
    valid_trips_with_dates
    .filter(F.col("bike_id") == max_bike_id)
    .select(
        "trip_id",
        "bike_id",
        "start_ts",
        "end_ts",
        "duration",
        "start_station_id",
        "start_station_name",
        "end_station_id",
        "end_station_name",
    )
    .orderBy("start_ts", "trip_id")
)

max_bike_route.show(50, truncate=False)

bike_id with maximum total duration: 535
+-------+-------+-------------------+-------------------+--------+----------------+---------------------------------------------+--------------+----------------------------------------+
|trip_id|bike_id|start_ts           |end_ts             |duration|start_station_id|start_station_name                           |end_station_id|end_station_name                        |
+-------+-------+-------------------+-------------------+--------+----------------+---------------------------------------------+--------------+----------------------------------------+
|4966   |535    |2013-08-29 19:32:00|2013-08-29 19:53:00|1245    |47              |Post at Kearney                              |70            |San Francisco Caltrain (Townsend at 4th)|
|5067   |535    |2013-08-29 21:38:00|2013-08-29 21:45:00|423     |70              |San Francisco Caltrain (Townsend at 4th)     |69            |San Francisco Caltrain 2 (330 Townsend) |
|5179   |535    |2013-08-30 0

In [10]:
route_summary = (
    max_bike_route
    .select(
        F.concat_ws(" -> ", F.col("start_station_name"), F.col("end_station_name")).alias("route_segment")
    )
    .limit(30)
)

route_summary.show(30, truncate=False)

+-----------------------------------------------------------------------------------+
|route_segment                                                                      |
+-----------------------------------------------------------------------------------+
|Post at Kearney -> San Francisco Caltrain (Townsend at 4th)                        |
|San Francisco Caltrain (Townsend at 4th) -> San Francisco Caltrain 2 (330 Townsend)|
|San Francisco Caltrain 2 (330 Townsend) -> Market at Sansome                       |
|Market at Sansome -> 2nd at South Park                                             |
|2nd at Townsend -> Davis at Jackson                                                |
|San Francisco City Hall -> Civic Center BART (7th at Market)                       |
|Civic Center BART (7th at Market) -> Post at Kearney                               |
|Post at Kearney -> Embarcadero at Sansome                                          |
|Embarcadero at Sansome -> Washington at Kearney      

## Task 4. Number of bikes in the system

Count distinct `bike_id` values.

In [11]:
bikes_count = valid_trips.select("bike_id").distinct().count()
print("Number of bikes in the system:", bikes_count)

Number of bikes in the system: 700


## Task 5. Users who spent more than 3 hours on trips

There is no real `user_id` field in the dataset. The available user-like identifier is `zip_code`, so it is used here as the user key.
3 hours = 10800 seconds.

In [12]:
users_over_3_hours = (
    valid_trips
    .withColumn("zip_code", F.trim(F.col("zip_code").cast("string")))
    .filter(F.col("zip_code").rlike(r"^[0-9]{5}$"))
    .groupBy("zip_code")
    .agg(
        F.sum("duration").alias("total_duration_sec"),
        F.count("*").alias("trips_count"),
    )
    .withColumn("total_duration_hours", F.round(F.col("total_duration_sec") / 3600, 2))
    .filter(F.col("total_duration_sec") > 3 * 60 * 60)
    .orderBy(F.desc("total_duration_sec"))
)

users_over_3_hours.show(50, truncate=False)
print("Number of zip_code/user values over 3 hours:", users_over_3_hours.count())

+--------+------------------+-----------+--------------------+
|zip_code|total_duration_sec|trips_count|total_duration_hours|
+--------+------------------+-----------+--------------------+
|94107   |49757162          |78704      |13821.43            |
|94105   |25596128          |42672      |7110.04             |
|94133   |21637675          |31359      |6010.47             |
|94102   |19128021          |19757      |5313.34             |
|94103   |19127388          |26673      |5313.16             |
|95531   |17270400          |1          |4797.33             |
|94111   |14244997          |21409      |3956.94             |
|95112   |12742370          |11564      |3539.55             |
|94109   |12057128          |13989      |3349.2              |
|94040   |7807926           |7114       |2168.87             |
|94110   |7421936           |7621       |2061.65             |
|94117   |6901313           |9851       |1917.03             |
|94301   |6590378           |3868       |1830.66       

## Summary

All 5 tasks are solved in this notebook. Task 5 uses `zip_code` because `trips.csv` does not contain a real `user_id` field.

In [13]:
spark.stop()